In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tensorflow.keras.layers import LSTM, Dense, Masking, Input, Embedding, Concatenate, LayerNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy
import keras
keras.utils.set_random_seed(28)
import datetime
from icecream import ic
from itertools import product
from sklearn.preprocessing import MinMaxScaler
import wandb
from configparser import ConfigParser
from sklearn.model_selection import train_test_split
from typing import Iterable

In [2]:
CUSTOM_DATASET_PATH = Path("custom_data/")

In [3]:
merged = pd.read_csv(CUSTOM_DATASET_PATH / "merged_2.csv")
merged

,business_id,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,Brasseries_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
2,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,3,8,2,2,5,0,18
3,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
4,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
511134,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
511135,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
511136,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


As the average stars review is 3.86, we will base our recommender only on positive reviews (at least 4 stars), as it is not our goal to only predict the next restaurants but also the next "good" restaurant. 

In [4]:
merged = merged[merged["stars_review"] >= 4]

We are filtering out users who only have one review as we need at least a 2 restaurant long sequence for our x and y split.

In [5]:
user_ids_to_filter_out = merged.value_counts("user_id")[merged.value_counts("user_id") < 2].index
user_ids_to_filter_out

Index(['zzx7J3zheFF3zf5YYfDAMg', '--9SwR3jQX-e3opxARFF_g',
       '--B0My3yotQA2kLVWz6IYA', 'zymqiN-cBSAiZqsbs6TohQ',
       'zyjzkLBrYJO5mYQ7sEGhNA', '-00bkUAlXw9Z6SB-B6SBLA',
       '--z4j_95J4XLJJFrVHgBiw', '--t1GgwabT-J_6OQG8f5QQ',
       '--mQ4S5h1tXzvE9VDYVwdQ', '--m6DE1KNxjSyL6LFYYrYQ',
       ...
       '--a-gJTBfWStlOFvVpcMMw', '--YWBD-491bjkqJWEUGaDg',
       '--Whu2Ivi3IXXBo7aU1i3A', '--UizzbnQlZg7bEv2oXEyg',
       '--UhENQdbuWEh0mU5weIEg', '--S8M395r8NtOCvS2LRfDw',
       '--FxiKq-x-zu-tfn4F55DA', '--FcqRehCCnPWDzTpKyO0Q',
       '--Dz7-yZ5vMuDdlHxfBWVw', '--Dq76pp192-Wk-vo_dPmg'],
      dtype='object', name='user_id', length=86467)

In [6]:
merged = merged[~ merged["user_id"].isin(user_ids_to_filter_out)]
merged.reset_index(inplace=True)
merged

,index,business_id,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,1,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
1,3,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
2,4,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
3,5,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,4,0,0,41,34,51,51,26,97,669
4,7,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,5,2,0,45,114,129,129,63,81,834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272828,511132,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,0,0,0,0,0,1,3
272829,511133,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
272830,511134,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
272831,511136,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [7]:
# encoding business_id string to an integer
merged["business_id"], _ = pd.factorize(merged["business_id"])
# zero will be used as padding
merged["business_id"] += 1

In [8]:
merged

,index,business_id,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,1,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
1,3,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
2,4,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
3,5,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,4,0,0,41,34,51,51,26,97,669
4,7,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,5,2,0,45,114,129,129,63,81,834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272828,511132,3432,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,0,0,0,0,0,1,3
272829,511133,3432,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
272830,511134,3432,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
272831,511136,3432,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [9]:
# encoding user_id string to an integer
merged["user_id"], _ = pd.factorize(merged["user_id"])

In [10]:
# merged.info(max_cols=500)

## MinMax Scaling
We will divide the dataframe into parts that are to be minmaxed (categories, attributes) and the rest (ids).

In [11]:
merged_no_ids = merged.drop(labels=["review_id", "user_id", "business_id"], axis=1)
display(merged_no_ids)

,index,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,Brasseries_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,1,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
1,3,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
2,4,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
3,5,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,4,0,0,41,34,51,51,26,97,669
4,7,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,5,2,0,45,114,129,129,63,81,834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272828,511132,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,3
272829,511133,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
272830,511134,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
272831,511136,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [12]:
merged_only_ids = merged[["review_id", "user_id", "business_id"]]
merged_only_ids

,review_id,user_id,business_id
0,uduvUCvi9w3T2bSGivCfXg,0,1
1,MKNp_CdR2k2202-c8GN5Dw,1,1
2,D1GisLDPe84Rrk_R4X2brQ,2,1
3,_hJu0u6nB-8LIeQJY4Vg4w,3,1
4,_xRGsS6QGpcD9LQJNtavuw,4,1
...,...,...,...
272828,cOh-a-xWgOBP4WHxVp2SOg,34500,3432
272829,qBcwQEQPnLxjkw-xbUIF4Q,33572,3432
272830,G8fbysnUAUmqq1XWTjMQ4Q,17780,3432
272831,JITY01bGbdsiUBznLz9rdg,7973,3432


In [13]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(merged_no_ids)
merged_no_ids = pd.DataFrame(scaler.transform(merged_no_ids), columns=merged_no_ids.columns, index=merged_no_ids.index)
display(merged_no_ids.head())

,index,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,Brasseries_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,0.000000,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.001908,0.000783,0.000000,0.001863,0.001154,0.001959,0.001959,0.003021,0.000709,0.012338
1,0.000004,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
2,0.000006,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000783,0.003433,0.004064,0.002102,0.001491,0.001491,0.001510,0.000177,0.009337
3,0.000008,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.001272,0.000000,0.000000,0.006942,0.001401,0.002172,0.002172,0.003021,0.003439,0.044551
4,0.000012,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.001590,0.001566,0.000000,0.007619,0.004699,0.005495,0.005495,0.007320,0.002872,0.055556


In [14]:
merged = merged_only_ids.join(merged_no_ids)
display(merged)

,review_id,user_id,business_id,index,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,uduvUCvi9w3T2bSGivCfXg,0,1,0.000000,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.001908,0.000783,0.000000,0.001863,0.001154,0.001959,0.001959,0.003021,0.000709,0.012338
1,MKNp_CdR2k2202-c8GN5Dw,1,1,0.000004,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
2,D1GisLDPe84Rrk_R4X2brQ,2,1,0.000006,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.000000,0.000783,0.003433,0.004064,0.002102,0.001491,0.001491,0.001510,0.000177,0.009337
3,_hJu0u6nB-8LIeQJY4Vg4w,3,1,0.000008,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.001272,0.000000,0.000000,0.006942,0.001401,0.002172,0.002172,0.003021,0.003439,0.044551
4,_xRGsS6QGpcD9LQJNtavuw,4,1,0.000012,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.001590,0.001566,0.000000,0.007619,0.004699,0.005495,0.005495,0.007320,0.002872,0.055556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272828,cOh-a-xWgOBP4WHxVp2SOg,34500,3432,0.999990,0.242793,0.505308,0.875,0.005248,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000035,0.000133
272829,qBcwQEQPnLxjkw-xbUIF4Q,33572,3432,0.999992,0.242793,0.505308,0.875,0.005248,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000041,0.000000,0.000000,0.000000,0.000000,0.000000
272830,G8fbysnUAUmqq1XWTjMQ4Q,17780,3432,0.999994,0.242793,0.505308,0.875,0.005248,0.0,0.0,...,0.000000,0.000000,0.000000,0.000508,0.000124,0.000170,0.000170,0.000116,0.000106,0.000333
272831,JITY01bGbdsiUBznLz9rdg,7973,3432,0.999998,0.242793,0.505308,0.875,0.005248,0.0,0.0,...,0.000000,0.000000,0.000000,0.000847,0.000000,0.000000,0.000000,0.000000,0.000000,0.005069


## Splitting into sequences

In [15]:
sequence_length = 5

### Less than sequence length

In [16]:
def split_to_sequences_padding(
        df: pd.DataFrame,
        user_indices: Iterable,
        sequence_length: int,
        masking_value: int = -1
        ) -> list[pd.DataFrame]:
    """Split dataframe to sequences.

    Splits dataframe to time-series sequences based on user_id. Applies padding
    with masking_value to achieve the desired sequence_length. Business id is
    to be masked with zero (because of keras).

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to be split into sequences.
    user_indices : Iterable
        Indices to be used what users are to be split into sequences.
    sequence_length : int
        The desired sequence length.
    masking_value : int, optional 
        The value to be used in padded restaurant reviews. By default -1.

    Returns
    -------
    list[pd.DataFrame]
        Return a list of sequences in form of a dataframe.
    """
    user_sequences = []
    for user_id in user_indices:
        user_df = df[df["user_id"] == user_id]
        # the first rows containing the "question"
        first_rows = user_df.head(n=len(user_df)-1)

        # the row containing the answer (predicted restaurant)
        last_row = user_df.tail(n=1)
        to_pad_length = sequence_length - len(user_df)
        to_pad = user_df.head(n=1)
        to_pad = pd.concat(to_pad_length * [to_pad])
        float_columns = to_pad.select_dtypes(include="float64").columns
        # zero is used for padding of restaurant ids
        to_pad["business_id"] = 0
        to_pad.loc[:, float_columns] = masking_value

        user_sequence = pd.concat([first_rows, to_pad, last_row])
        user_sequences.append(user_sequence)
    return user_sequences

In [17]:
indexes_less_than_sequence_length = merged.value_counts("user_id")[(merged.value_counts("user_id") > 1) & (merged.value_counts("user_id") < sequence_length)].index

In [18]:
user_sequences_padding = split_to_sequences_padding(df=merged, user_indices=indexes_less_than_sequence_length, sequence_length=sequence_length)

In [19]:
print("len(user_sequences_padding):", len(user_sequences_padding))

len(user_sequences_padding): 36176


In [20]:
user_matrix_padding = np.array(
    [user_df.drop(columns=["review_id"]).to_numpy() for user_df in user_sequences_padding]
)

In [21]:
np.save(CUSTOM_DATASET_PATH / "user_matrix_padding_5", user_matrix_padding, allow_pickle=False)

In [22]:
user_matrix_padding = np.load(CUSTOM_DATASET_PATH / "user_matrix_padding_5.npy")

In [23]:
print("user_matrix_padding.shape:", user_matrix_padding.shape)

user_matrix_padding.shape: (36176, 5, 385)


### More or equal to sequence length

In [24]:
def split_to_sequences(
        df: pd.DataFrame,
        user_indices: Iterable,
        sequence_length: int
        ) -> tuple[list[pd.DataFrame], pd.DataFrame]:
    """
    Split dataframe into sequences.

    Splits dataframe to time-series sequences based on user_id. Assumed that all
    users have the sequence length of at least the sequence_length. Sequences
    that go over the limit are split and returned in a dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to be split into sequences.
    user_indices : Iterable
        Indices to be used what users are to be split into sequences.
    sequence_length : int
        The desired sequence length.

    Returns
    -------
    list[pd.DataFrame]
        Return a list of sequences in form of a dataframe.
    pd.DataFrame
        The rest of user sequences. They are all shorter than the sequence
        length.
    """

    user_sequences = []
    rest_of_user_sequences = []
    for user_id in user_indices:
        user_df = df[df["user_id"] == user_id].sort_values("date")

        while len(user_df) >= sequence_length:
            user_sequences.append(user_df.iloc[:sequence_length])
            user_df = user_df.iloc[sequence_length:]
        rest_of_user_sequences.append(user_df)

    return user_sequences, pd.concat(rest_of_user_sequences)

In [25]:
indexes = merged.value_counts("user_id")[(merged.value_counts("user_id") >= sequence_length)].index

In [26]:
user_sequences, rest_of_user_sequences = split_to_sequences(merged, user_indices=indexes, sequence_length=5)

In [27]:
user_matrix = np.array(
    [user_df.drop(columns=["review_id"]).to_numpy() for user_df in user_sequences]
)

In [28]:
np.save(CUSTOM_DATASET_PATH / "user_matrix_5", user_matrix, allow_pickle=False)

In [29]:
user_matrix = np.load(CUSTOM_DATASET_PATH / "user_matrix_5.npy")

In [30]:
print("user_matrix.shape:", user_matrix.shape)

user_matrix.shape: (32052, 5, 385)


### The rest of more or equal to sequence length

In [31]:
rest_of_user_sequences

,review_id,user_id,business_id,index,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
22234,AFNqh6PvrZpHrzr_Mh5gpA,2930,299,0.079638,0.284109,0.508034,0.875,0.008397,0.0,0.0,...,0.000318,0.000000,0.0,0.011514,0.003669,0.001832,0.001832,0.003369,0.001666,0.009804
182138,xRMMNUcdRRbuzCmwDGFokg,2930,2255,0.668941,0.152727,0.527117,0.375,0.001225,0.0,0.0,...,0.000318,0.000000,0.0,0.011514,0.003669,0.001832,0.001832,0.003369,0.001666,0.009804
98239,XKkWOMNPKdAJs99CD3mvYA,293,1193,0.355600,0.622611,0.361358,1.000,0.008572,0.0,0.0,...,0.007952,0.000000,0.0,0.015069,0.010841,0.014611,0.014611,0.011502,0.010211,0.023543
148321,j8AMTLpOZmkP5IqqUd8zKQ,293,1791,0.544499,0.662821,0.322732,0.625,0.015045,0.0,0.0,...,0.007952,0.000000,0.0,0.015069,0.010841,0.014611,0.014611,0.011502,0.010211,0.023543
188847,NV_WYM_eRw8Xy_HUUvG1pg,293,2346,0.692102,0.344865,0.539596,0.875,0.006823,0.0,0.0,...,0.007952,0.000000,0.0,0.015069,0.010841,0.014611,0.014611,0.011502,0.010211,0.023543
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170441,eOtCTEb54LvMDMmFeLlqIw,22208,2125,0.627050,0.787810,0.382791,0.625,0.029391,0.0,0.0,...,0.000000,0.000000,0.0,0.000169,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
123591,n-xBCRJ2IztsZ6T_B7BpyA,22169,1487,0.454897,0.302656,0.465449,0.750,0.153779,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000067
138229,87xloxxhNIIqQk509SnW-g,22176,1651,0.506343,0.316116,0.513624,0.750,0.211337,0.0,0.0,...,0.000000,0.000783,0.0,0.000339,0.000041,0.000000,0.000000,0.000116,0.000000,0.021475
228116,aWLDcv9deASjhNBBN3IZPQ,34,2871,0.840860,0.379121,0.540243,0.750,0.053184,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [32]:
indexes_less_than_sequence_length = rest_of_user_sequences.value_counts("user_id")[(rest_of_user_sequences.value_counts("user_id") > 1) & (rest_of_user_sequences.value_counts("user_id") < sequence_length)].index

In [33]:
rest_of_user_sequences_padding = split_to_sequences_padding(df=rest_of_user_sequences, user_indices=indexes_less_than_sequence_length, sequence_length=sequence_length)

In [34]:
rest_of_user_matrix_padding = np.array(
    [user_df.drop(columns=["review_id"]).to_numpy() for user_df in rest_of_user_sequences_padding]
)

In [35]:
np.save(CUSTOM_DATASET_PATH / "rest_of_user_matrix_padding_5", rest_of_user_matrix_padding, allow_pickle=False)

In [36]:
rest_of_user_matrix_padding = np.load(CUSTOM_DATASET_PATH / "rest_of_user_matrix_padding_5.npy")

In [37]:
print("rest_of_user_matrix_padding.shape:", rest_of_user_matrix_padding.shape)

rest_of_user_matrix_padding.shape: (6007, 5, 385)


## Merging all the sequences

In [38]:
# merge all the matrices
final_user_matrix = np.concatenate([user_matrix, user_matrix_padding, rest_of_user_matrix_padding])
print("final_user_matrix.shape:", final_user_matrix.shape)

final_user_matrix.shape: (74235, 5, 385)


In [39]:
user_matrix_x = final_user_matrix[:, :sequence_length-1, :]
print("user_matrix_x.shape:", user_matrix_x.shape)

user_matrix_x.shape: (74235, 4, 385)


In [40]:
user_ids_x = user_matrix_x[:, :, 0]
restaurant_ids_x = user_matrix_x[:, :, 1]
attributes_x = user_matrix_x[:, :, 2:]
print("user_ids_x.shape:", user_ids_x.shape)
print("restaurant_ids_x.shape:", restaurant_ids_x.shape)
print("attributes_x.shape:", attributes_x.shape)

user_ids_x.shape: (74235, 4)
restaurant_ids_x.shape: (74235, 4)
attributes_x.shape: (74235, 4, 383)


In [41]:
user_matrix_y = final_user_matrix[:, sequence_length-1:, 1:2]
print("user_matrix_y.shape:", user_matrix_y.shape)

user_matrix_y.shape: (74235, 1, 1)


## Train/Test split

In [42]:
user_ids_x_train, user_ids_x_test, restaurant_ids_x_train, restaurant_ids_x_test, attributes_x_train, attributes_x_test, y_train, y_test = train_test_split(user_ids_x, restaurant_ids_x, attributes_x, user_matrix_y, train_size=0.8, random_state=28)

In [43]:
print("user_ids_x_train.shape:", user_ids_x_train.shape)
print("user_ids_x_train.dtype:", user_ids_x_train.dtype)
print("user_ids_x_test.shape:", user_ids_x_test.shape)
print("user_ids_x_test.dtype:", user_ids_x_test.dtype)
print("restaurant_ids_x_train.shape:", restaurant_ids_x_train.shape)
print("restaurant_ids_x_train.dtype:", restaurant_ids_x_train.dtype)
print("restaurant_ids_x_test.shape:", restaurant_ids_x_test.shape)
print("restaurant_ids_x_test.dtype:", restaurant_ids_x_test.dtype)
print("attributes_x_train.shape:", attributes_x_train.shape)
print("attributes_x_train.dtype:", attributes_x_train.dtype)
print("attributes_x_test.shape:", attributes_x_test.shape)
print("attributes_x_test.dtype:", attributes_x_test.dtype)
print("y_train.shape:", y_train.shape)
print("y_train.dtype:", y_train.dtype)
print("y_test.shape:", y_test.shape)
print("y_test.dtype:", y_test.dtype)

user_ids_x_train.shape: (59388, 4)
user_ids_x_train.dtype: float64
user_ids_x_test.shape: (14847, 4)
user_ids_x_test.dtype: float64
restaurant_ids_x_train.shape: (59388, 4)
restaurant_ids_x_train.dtype: float64
restaurant_ids_x_test.shape: (14847, 4)
restaurant_ids_x_test.dtype: float64
attributes_x_train.shape: (59388, 4, 383)
attributes_x_train.dtype: float64
attributes_x_test.shape: (14847, 4, 383)
attributes_x_test.dtype: float64
y_train.shape: (59388, 1, 1)
y_train.dtype: float64
y_test.shape: (14847, 1, 1)
y_test.dtype: float64


In [44]:
config = ConfigParser()
config.read("credentials.ini")
wandb_api_key = config["WandB"]["api_key"]

In [45]:
# hyperparameters and stuff
name = "recommendLayerNorm"
version = 12
learning_rate = 0.00001
number_of_epochs = 85
batch_size = 1024
user_embedding_dim = 28
restaurant_embedding_dim = 64
lstm_out_units = 112
logged = False

In [46]:
restaurant_input = Input(shape=(sequence_length-1,), dtype='float64', name='restaurant_ids')
user_input = Input(shape=(sequence_length-1,), dtype='float64', name='user_ids')
attributes_input = Input(shape=(sequence_length-1, attributes_x_train.shape[-1]), dtype='float64', name='attributes')

In [47]:
restaurants_ids = final_user_matrix[:, :, 1]
num_restaurants = int(restaurants_ids.max()) + 1 # zero is padding

In [48]:
user_ids = final_user_matrix[:, :, 0]
num_users = len(np.unique(user_ids))

In [49]:
restaurant_embedding = Embedding(
        input_dim=num_restaurants,
        output_dim=restaurant_embedding_dim,
        mask_zero=True,
        name='restaurant_embedding'
    )(restaurant_input)

user_embedding = Embedding(
        input_dim=num_users,
        output_dim=user_embedding_dim,
        name='user_embedding'
    )(user_input)

combined = Concatenate(axis=-1)([
        restaurant_embedding,
        user_embedding,
        attributes_input,
    ])

lstm_out = LSTM(lstm_out_units, return_sequences=False)(combined)
lstm_out_layer_norm = LayerNormalization()(lstm_out)
output = Dense(num_restaurants, activation='softmax')(lstm_out_layer_norm)

In [50]:
model = keras.Model(
    inputs=[restaurant_input, user_input, attributes_input],
    outputs=output
)

optimizer = keras.optimizers.Adam()
optimizer.learning_rate.assign(learning_rate)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy',
        SparseTopKCategoricalAccuracy(k=5, name='top_5_acc'),
        SparseTopKCategoricalAccuracy(k=10, name='top_10_acc')
    ]
)

model_summary = ""
def save_summary(line):
    global model_summary
    model_summary = model_summary + line

model.summary(print_fn=save_summary)

In [51]:
print(model_summary)

Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ restaurant_ids      │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ restaurant_embeddi… │ (None, 4, 64)     │    219,712 │ restaurant_ids[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_ids            │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attributes          │ (None, 4, 383)    │          0 │

In [52]:
class ModelCallback(keras.callbacks.Callback):
    def __init__(self, restaurant_ids_x_test, user_ids_x_test, attributes_x_test, y_test, batch_size, logged=True):
        self.x_test = [restaurant_ids_x_test, user_ids_x_test, attributes_x_test]
        self.y_test = y_test
        self.batch_size = batch_size
        self.logged = logged

    def on_epoch_end(self, epoch, logs=None):
        new_logs = dict()
        for metric in logs:
            new_logs["train_" + metric] = logs[metric]
        evaluation = self.model.evaluate(x=self.x_test, y=self.y_test, batch_size=self.batch_size)
        loss = evaluation[0]
        accuracy = evaluation[1]
        top_5_accuracy = evaluation[2]
        top_10_accuracy = evaluation[3]
        new_logs["test_loss"] = loss
        new_logs["test_accuracy"] = accuracy
        new_logs["test_top_5_accuracy"] = top_5_accuracy
        new_logs["test_top_10_accuracy"] = top_10_accuracy
        if self.logged:
            run.log(new_logs)

model_callback = ModelCallback(
    restaurant_ids_x_test=restaurant_ids_x_test,
    user_ids_x_test=user_ids_x_test,
    attributes_x_test=attributes_x_test,
    y_test=y_test,
    batch_size=batch_size,
    logged=logged
)

In [53]:
if logged:
    print("Logging = On!")
    run = wandb.init(
        entity="ISA2026",
        project="ISA_LSTM_2",
        name=name + "." + str(version),
        config={
            "learning_rate": learning_rate,
            "number_of_epochs": number_of_epochs,
            "batch_size": batch_size,
            "user_embedding_dim": user_embedding_dim,
            "restaurant_embedding_dim": restaurant_embedding_dim,
            "model_summary": model_summary,
            "lstm_out_units": lstm_out_units,
        },
    )
    run.log_code(".", include_fn=lambda path: path.endswith(".ipynb"))
else:
    print("Logging = Off!")

Logging = Off!


In [54]:
history = model.fit(
    [restaurant_ids_x_train, user_ids_x_train, attributes_x_train],
    y_train,
    batch_size=batch_size,
    epochs=number_of_epochs,
    verbose=2,
    callbacks=[model_callback]
)
if logged:
    run.finish()

Epoch 1/85
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 2.0621e-04 - loss: 8.1321 - top_10_acc: 0.0028 - top_5_acc: 0.0014
58/58 - 5s - 80ms/step - accuracy: 1.5336e-04 - loss: 8.1559 - top_10_acc: 0.0028 - top_5_acc: 0.0014
Epoch 2/85
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0136 - loss: 8.0905 - top_10_acc: 0.0325 - top_5_acc: 0.0284
58/58 - 3s - 48ms/step - accuracy: 0.0016 - loss: 8.1123 - top_10_acc: 0.0223 - top_5_acc: 0.0169
Epoch 3/85
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0175 - loss: 8.0487 - top_10_acc: 0.0393 - top_5_acc: 0.0327
58/58 - 3s - 45ms/step - accuracy: 0.0170 - loss: 8.0694 - top_10_acc: 0.0355 - top_5_acc: 0.0299
Epoch 4/85
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0177 - loss: 8.0067 - top_10_acc: 0.0438 - top_5_acc: 0.0350
58/58 - 3s - 45ms/step - accuracy: 0.0177 - loss: 8.0263 - top_10_acc: 0.0402 - top_5_acc: 0.0332
Epoch 5/85
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0180 - loss: 7.9638 - top_10_ac

## Naive Bayes

In [56]:
# Naive Bayes baseline for unordered interaction signals.
# The same train/test split is reused to keep LSTM and NB metrics directly comparable.
from scipy.sparse import csr_matrix, hstack
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, top_k_accuracy_score, log_loss
# Keep attributes optional: disabling them makes NB much faster on large data.
NB_USE_ATTRIBUTE_MEAN = False
def build_nb_features_sparse(
    restaurant_ids_seq: np.ndarray,
    attributes_seq: np.ndarray,
    num_items: int,
    use_attribute_mean: bool = False,
):
    """Build unordered sparse features for Naive Bayes.
    Feature blocks:
    - bag-of-restaurants counts (no order information),
    - history length (activity level),
    - optional mean review attributes over non-padded steps.
    """
    restaurant_ids_seq = restaurant_ids_seq.astype(np.int64)
    n_samples, seq_len = restaurant_ids_seq.shape
    # Build sparse count matrix of visited restaurants in the history window.
    row_idx = np.repeat(np.arange(n_samples), seq_len)
    col_idx = restaurant_ids_seq.reshape(-1)
    non_padding = col_idx != 0
    restaurant_counts = csr_matrix(
        (
            np.ones(non_padding.sum(), dtype=np.float32),
            (row_idx[non_padding], col_idx[non_padding]),
        ),
        shape=(n_samples, num_items + 1),
        dtype=np.float32,
    )
    # Column 0 is padding id, so it is dropped from the feature matrix.
    feature_blocks = [restaurant_counts[:, 1:]]
    valid_mask = (restaurant_ids_seq != 0)
    history_length = valid_mask.sum(axis=1, keepdims=True).astype(np.float32)
    feature_blocks.append(csr_matrix(history_length))
    if use_attribute_mean:
        valid_count = np.maximum(valid_mask.sum(axis=1, keepdims=True), 1)
        attributes_sum = (attributes_seq * valid_mask[..., None]).sum(axis=1)
        attributes_mean = (attributes_sum / valid_count).astype(np.float32)
        feature_blocks.append(csr_matrix(attributes_mean))
    return hstack(feature_blocks, format="csr", dtype=np.float32)
# Build train/test features from the same split used by the LSTM section.
X_train_nb = build_nb_features_sparse(
    restaurant_ids_seq=restaurant_ids_x_train,
    attributes_seq=attributes_x_train,
    num_items=num_restaurants,
    use_attribute_mean=NB_USE_ATTRIBUTE_MEAN,
)
X_test_nb = build_nb_features_sparse(
    restaurant_ids_seq=restaurant_ids_x_test,
    attributes_seq=attributes_x_test,
    num_items=num_restaurants,
    use_attribute_mean=NB_USE_ATTRIBUTE_MEAN,
)
# Target business ids are already integer encoded.
y_train_nb = y_train.reshape(-1).astype(np.int64)
y_test_nb = y_test.reshape(-1).astype(np.int64)
# MultinomialNB is a better match for sparse count-like features and is faster.
nb_model = MultinomialNB(alpha=0.3)
nb_model.fit(X_train_nb, y_train_nb)
nb_proba = nb_model.predict_proba(X_test_nb)
nb_classes = nb_model.classes_
nb_pred = nb_classes[np.argmax(nb_proba, axis=1)]
# Some test labels may be unseen in training due to the random split.
seen_class_mask = np.isin(y_test_nb, nb_classes)
y_test_seen = y_test_nb[seen_class_mask]
nb_proba_seen = nb_proba[seen_class_mask]
nb_pred_seen = nb_pred[seen_class_mask]
k5 = min(5, nb_proba.shape[1])
k10 = min(10, nb_proba.shape[1])
if y_test_seen.size > 0:
    test_log_loss = log_loss(y_test_seen, nb_proba_seen, labels=nb_classes)
    test_accuracy = accuracy_score(y_test_seen, nb_pred_seen)
    test_top_5_accuracy = top_k_accuracy_score(y_test_seen, nb_proba_seen, k=k5, labels=nb_classes)
    test_top_10_accuracy = top_k_accuracy_score(y_test_seen, nb_proba_seen, k=k10, labels=nb_classes)
else:
    test_log_loss = np.nan
    test_accuracy = np.nan
    test_top_5_accuracy = np.nan
    test_top_10_accuracy = np.nan
nb_metrics = pd.DataFrame([
    {
        "model": "naive_bayes_unordered",
        "features": "sparse_restaurant_counts" + ("+attribute_mean" if NB_USE_ATTRIBUTE_MEAN else "+history_length"),
        "test_log_loss": test_log_loss,
        "test_accuracy": test_accuracy,
        "test_top_5_accuracy": test_top_5_accuracy,
        "test_top_10_accuracy": test_top_10_accuracy,
        "n_classes_seen_in_train": int(len(nb_classes)),
        "n_test_samples": int(y_test_nb.size),
        "n_test_samples_seen_labels": int(y_test_seen.size),
        "n_test_samples_unseen_labels": int((~seen_class_mask).sum()),
        "seen_label_coverage": float(seen_class_mask.mean()),
    }
])
# Preview predicted top-k restaurants for a quick sanity check.
preview_k = min(5, nb_proba.shape[1])
topk_idx = np.argsort(-nb_proba, axis=1)[:, :preview_k]
topk_business_ids = nb_classes[topk_idx]
topk_scores = np.take_along_axis(nb_proba, topk_idx, axis=1)
nb_preview = pd.DataFrame({
    "true_business_id": y_test_nb[:10],
    "predicted_business_id_top1": nb_pred[:10],
    "top5_business_ids": [list(row) for row in topk_business_ids[:10]],
    "top5_scores": [list(np.round(row, 4)) for row in topk_scores[:10]],
    "label_seen_in_train": [bool(v) for v in seen_class_mask[:10]],
})
display(nb_metrics)
display(nb_preview)


,model,features,test_log_loss,test_accuracy,test_top_5_accuracy,test_top_10_accuracy,n_classes_seen_in_train,n_test_samples,n_test_samples_seen_labels,n_test_samples_unseen_labels,seen_label_coverage
0,naive_bayes_unordered,sparse_restaurant_counts+history_length,9.583206,0.031324,0.079056,0.113567,2970,14847,14749,98,0.993399


,true_business_id,predicted_business_id_top1,top5_business_ids,top5_scores,label_seen_in_train
0,2412,2994,"[2994, 3408, 2612, 2498, 3140]","[0.2067, 0.1836, 0.0584, 0.0433, 0.0426]",True
1,2162,1489,"[1489, 2498, 3138, 2355, 3245]","[0.3764, 0.135, 0.0701, 0.0353, 0.0307]",True
2,3421,3245,"[3245, 3181, 489, 340, 3138]","[0.1195, 0.0269, 0.019, 0.0166, 0.0131]",True
3,2635,3245,"[3245, 3408, 2989, 2612, 2498]","[0.0755, 0.0516, 0.0466, 0.0408, 0.0352]",True
4,1980,3360,"[3360, 2650, 3408, 3340, 2498]","[0.0516, 0.0484, 0.0414, 0.0408, 0.0406]",True
5,1283,1234,"[1234, 2498, 2260, 3245, 3140]","[0.4047, 0.2901, 0.1335, 0.032, 0.0246]",True
6,1488,3245,"[3245, 2989, 3340, 2498, 2951]","[0.5723, 0.1777, 0.0745, 0.0502, 0.0111]",True
7,2133,3245,"[3245, 3140, 489, 2498, 2612]","[0.6231, 0.0997, 0.0422, 0.0344, 0.0217]",True
8,2818,2989,"[2989, 2498, 3140, 1489, 3245]","[0.5017, 0.0935, 0.0727, 0.0639, 0.0526]",True
9,1471,1489,"[1489, 2951, 1234, 3340, 3245]","[0.5279, 0.161, 0.0449, 0.0315, 0.0311]",True


In [57]:
# Compare ordered (LSTM) vs unordered (Naive Bayes) on the exact same test split.
# LSTM keeps sequence order via recurrent modeling, NB uses aggregated order-free features.
lstm_eval = model.evaluate(
    [restaurant_ids_x_test, user_ids_x_test, attributes_x_test],
    y_test,
    batch_size=batch_size,
    verbose=0,
)
lstm_metrics = pd.DataFrame([
    {
        "model": "lstm_ordered",
        "representation": "ordered_sequence",
        "test_loss_or_log_loss": float(lstm_eval[0]),
        "test_accuracy": float(lstm_eval[1]),
        "test_top_5_accuracy": float(lstm_eval[2]),
        "test_top_10_accuracy": float(lstm_eval[3]),
    }
])
nb_for_compare = nb_metrics.rename(columns={"test_log_loss": "test_loss_or_log_loss"}).copy()
nb_for_compare["representation"] = "unordered_aggregated"
comparison_columns = [
    "model",
    "representation",
    "test_loss_or_log_loss",
    "test_accuracy",
    "test_top_5_accuracy",
    "test_top_10_accuracy",
]
comparison_df = pd.concat([
    lstm_metrics[comparison_columns],
    nb_for_compare[comparison_columns],
], ignore_index=True)
# Lower loss is better; higher accuracy/top-k values are better.
display(comparison_df)


,model,representation,test_loss_or_log_loss,test_accuracy,test_top_5_accuracy,test_top_10_accuracy
0,lstm_ordered,ordered_sequence,6.974335,0.029212,0.071934,0.106486
1,naive_bayes_unordered,unordered_aggregated,9.583206,0.031324,0.079056,0.113567
